In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm as scipy_norm

In [2]:
alpha        = 0.05
z_crit       = scipy_norm.ppf(1 - alpha / 2)
random_state = 42

all_models = [
    'gpt-5.2',
    'claude-sonnet-4.6',
    'claude-opus-4.6',
    'gemini-3.1-pro',
    'claude-haiku-4.5',
    'llama-3-8b',
    'mistral-large-2512',
    'gpt-oss-120b',
]

model_cols = {
    m: {
        'zs': f'{m} (zero shot prompting)',
        'fs': f'{m} (few shot)',
        'pp': f'{m} (persona prompt)',
    }
    for m in all_models
}

demo_cols = ['gender', 'race', 'age', 'education']

demographic_groups = {
    'Woman':           lambda df: df['gender'] == 'Woman',
    'Man':             lambda df: df['gender'] == 'Man',
    'White':           lambda df: df['race'] == 'White',
    'Black/Afr. Am.':  lambda df: df['race'] == 'Black or African American',
    'Hispanic/Latino': lambda df: df['race'] == 'Hispanic or Latino',
    'Asian':           lambda df: df['race'] == 'Asian',
    'Age 18-34':       lambda df: df['age'].isin(['18-24', '25-29', '30-34']),
    'Age 35-49':       lambda df: df['age'].isin(['35-39', '40-44', '45-49']),
    'Age 50+':         lambda df: df['age'].isin(['50-54', '54-59', '60-64', '>65']),
    'College degree':  lambda df: df['education'] == 'College degree',
    'HS diploma':      lambda df: df['education'] == 'High school diploma or equivalent',
    'Grad. degree':    lambda df: df['education'] == 'Graduate degree',
}
group_names = list(demographic_groups.keys())

# ── load both tasks ──────────────────────────────────────────────────────────
required_cols = [c for m in all_models for c in model_cols[m].values()]

def load_task(csv_path, label_col, threshold):
    df = (
        pd.read_csv(csv_path)
          .query("gender != 'Non-binary'")
          .dropna(subset=required_cols)
          .sample(frac=1, random_state=random_state)
          .reset_index(drop=True)
          .copy()
    )
    df['human_true'] = (df[label_col] >= threshold).astype(float)
    return df

data_pol = load_task('data/raw_data_llm_politeness.csv',   'politeness',    4)
data_off = load_task('data/raw_data_llm_ofensiveness.csv', 'offensiveness', 2)

for name, df in [('Politeness', data_pol), ('Offensiveness', data_off)]:
    print(f'{name}: N={len(df)}, base rate={df["human_true"].mean():.3f}')

Politeness: N=882, base rate=0.444
Offensiveness: N=801, base rate=0.320


In [3]:
def evaluate_llm(data, Hhat, true_thetas, group_masks):
    """
    Evaluate a single LLM prediction array on all demographic groups.

    Uses ALL N LLM predictions (no sampling) to measure:
      - Coverage : does the Bernoulli CI on the LLM mean cover true_theta?
      - Avg Delta: |LLM mean - true_theta| in percentage points

    Returns arrays of length n_groups (np.nan for empty groups).
    """
    n_groups = len(group_masks)
    coverage  = np.full(n_groups, np.nan)
    avg_delta = np.full(n_groups, np.nan)

    for g, (mask, true_theta) in enumerate(zip(group_masks, true_thetas)):
        if mask.sum() < 2:
            continue
        p   = Hhat[mask].mean()
        se  = np.sqrt(max(p * (1 - p), 1e-6) / mask.sum())
        lb  = p - z_crit * se
        ub  = p + z_crit * se
        coverage[g]  = float(lb <= true_theta <= ub)
        avg_delta[g] = abs(p - true_theta) * 100

    return coverage, avg_delta


def run_appendix_thresh(data, label, threshold):
    """Evaluate all 8 × 3 model-prompting combos on one dataset."""
    human_true   = data['human_true'].values
    group_masks  = [mask_fn(data).values for mask_fn in demographic_groups.values()]
    true_thetas  = np.array([human_true[m].mean() for m in group_masks])

    rows = []
    for model in all_models:
        cols = model_cols[model]
        missing = [c for c in cols.values() if c not in data.columns]
        if missing:
            print(f'  Skipping {model}: missing {missing}')
            continue

        variants = {
            'Zero-shot': (data[cols['zs']] >= threshold).astype(float).to_numpy(),
            'Few-shot':  (data[cols['fs']] >= threshold).astype(float).to_numpy(),
            'Persona':   (data[cols['pp']] >= threshold).astype(float).to_numpy(),
        }

        for vname, Hhat in variants.items():
            cov, delta = evaluate_llm(data, Hhat, true_thetas, group_masks)
            rows.append({
                'Task':       label,
                'Model':      model,
                'Prompting':  vname,
                'Coverage':   np.nanmean(cov),
                'Avg Delta':  np.nanmean(delta),
                **{f'cov_{g}':  cov[i]   for i, g in enumerate(group_names)},
                **{f'dlt_{g}':  delta[i] for i, g in enumerate(group_names)},
            })
    return pd.DataFrame(rows)


res_pol = run_appendix_thresh(data_pol, 'Politeness',    threshold=4)
res_off = run_appendix_thresh(data_off, 'Offensiveness', threshold=2)
results  = pd.concat([res_pol, res_off], ignore_index=True)

print(results[['Task', 'Model', 'Prompting', 'Coverage', 'Avg Delta']].to_string(index=False))

         Task              Model Prompting  Coverage  Avg Delta
   Politeness            gpt-5.2 Zero-shot  0.363636   6.496770
   Politeness            gpt-5.2  Few-shot  0.636364   6.550290
   Politeness            gpt-5.2   Persona  0.545455   6.424185
   Politeness  claude-sonnet-4.6 Zero-shot  0.090909  11.958886
   Politeness  claude-sonnet-4.6  Few-shot  0.272727  10.535428
   Politeness  claude-sonnet-4.6   Persona  0.636364   6.569775
   Politeness    claude-opus-4.6 Zero-shot  0.454545   8.173642
   Politeness    claude-opus-4.6  Few-shot  0.454545   9.119059
   Politeness    claude-opus-4.6   Persona  0.454545   8.277014
   Politeness     gemini-3.1-pro Zero-shot  0.545455   5.726442
   Politeness     gemini-3.1-pro  Few-shot  0.636364   7.724216
   Politeness     gemini-3.1-pro   Persona  0.545455   6.300169
   Politeness   claude-haiku-4.5 Zero-shot  0.000000  15.496417
   Politeness   claude-haiku-4.5  Few-shot  0.727273   5.953731
   Politeness   claude-haiku-4.5   Perso

/var/folders/gp/zvrqjcgn7sz1_bwmzn3kfwd40000gn/T/ipykernel_46461/1844003506.py:32: RuntimeWarning: Mean of empty slice.
  true_thetas  = np.array([human_true[m].mean() for m in group_masks])
/Users/kristinagligoric/miniconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/var/folders/gp/zvrqjcgn7sz1_bwmzn3kfwd40000gn/T/ipykernel_46461/1844003506.py:32: RuntimeWarning: Mean of empty slice.
  true_thetas  = np.array([human_true[m].mean() for m in group_masks])
/Users/kristinagligoric/miniconda3/lib/python3.12/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [4]:
def make_latex_table(results):
    """
    Build a LaTeX table* comparing all 8 models × 3 prompting strategies
    on both tasks.  Columns: Model | Prompting | Pol Cov | Pol Δ | Off Cov | Off Δ
    Coverage ≥ 0.90 is bolded.
    """
    pol = results[results['Task'] == 'Politeness'].set_index(['Model', 'Prompting'])
    off = results[results['Task'] == 'Offensiveness'].set_index(['Model', 'Prompting'])

    variant_order = ['Zero-shot', 'Few-shot', 'Persona']
    prompting_tex = {
        'Zero-shot': r'\textsc{zero-shot}',
        'Few-shot':  r'\textsc{few-shot}',
        'Persona':   r'\textsc{persona}',
    }

    cov_fmt = lambda c: r'\textbf{' + f'{c:.2f}' + r'}' if c >= 0.90 else f'{c:.2f}'
    dlt_fmt = lambda d: f'{d:.1f}'

    lines = [
        r'\begin{table*}[t]',
        r'\centering',
        r'\small',
        r'\begin{tabular}{llcccc}',
        r'\toprule',
        r'\multirow{2}{*}{\textbf{Model}} & \multirow{2}{*}{\textbf{Prompting}}'
        r' & \multicolumn{2}{c}{\textbf{Politeness}}'
        r' & \multicolumn{2}{c}{\textbf{Offensiveness}} \\',
        r'\cmidrule(lr){3-4}\cmidrule(lr){5-6}',
        r'& & Cov. & $\bar\Delta$ (\%pt) & Cov. & $\bar\Delta$ (\%pt) \\',
        r'\midrule',
    ]

    for m_i, model in enumerate(all_models):
        if m_i > 0:
            lines.append(r'\addlinespace[3pt]')
        for v_i, variant in enumerate(variant_order):
            try:
                pc  = pol.loc[(model, variant), 'Coverage']
                pd_ = pol.loc[(model, variant), 'Avg Delta']
                oc  = off.loc[(model, variant), 'Coverage']
                od  = off.loc[(model, variant), 'Avg Delta']
            except KeyError:
                continue
            model_str = r'\texttt{' + model + r'}' if v_i == 0 else ''
            lines.append(
                f'  {model_str} & {prompting_tex[variant]}'
                f' & {cov_fmt(pc)} & {dlt_fmt(pd_)}'
                f' & {cov_fmt(oc)} & {dlt_fmt(od)} \\\\'
            )

    lines += [
        r'\bottomrule',
        r'\end{tabular}',
        r'\caption{LLM-only evaluation across 8 models and 3 prompting strategies'
        r' on politeness and offensiveness annotation tasks.'
        r' Cov.\ = empirical 95\% CI coverage (target: 0.95; \textbf{bold} $\geq$ 0.90).'
        r' $\bar\Delta$ = mean absolute error in percentage points (lower is better),'
        r' averaged over all non-empty demographic subgroups.}',
        r'\label{tab:appendix_llm_eval}',
        r'\end{table*}',
    ]
    return '\n'.join(lines)


latex = make_latex_table(results)
print(latex)

\begin{table*}[t]
\centering
\small
\begin{tabular}{llcccc}
\toprule
\multirow{2}{*}{\textbf{Model}} & \multirow{2}{*}{\textbf{Prompting}} & \multicolumn{2}{c}{\textbf{Politeness}} & \multicolumn{2}{c}{\textbf{Offensiveness}} \\
\cmidrule(lr){3-4}\cmidrule(lr){5-6}
& & Cov. & $\bar\Delta$ (\%pt) & Cov. & $\bar\Delta$ (\%pt) \\
\midrule
  \texttt{gpt-5.2} & \textsc{zero-shot} & 0.36 & 6.5 & 0.09 & 18.7 \\
   & \textsc{few-shot} & 0.64 & 6.6 & 0.09 & 13.5 \\
   & \textsc{persona} & 0.55 & 6.4 & 0.09 & 18.4 \\
\addlinespace[3pt]
  \texttt{claude-sonnet-4.6} & \textsc{zero-shot} & 0.09 & 12.0 & 0.00 & 37.7 \\
   & \textsc{few-shot} & 0.27 & 10.5 & 0.64 & 6.0 \\
   & \textsc{persona} & 0.64 & 6.6 & 0.00 & 41.4 \\
\addlinespace[3pt]
  \texttt{claude-opus-4.6} & \textsc{zero-shot} & 0.45 & 8.2 & 0.45 & 8.0 \\
   & \textsc{few-shot} & 0.45 & 9.1 & 0.27 & 13.1 \\
   & \textsc{persona} & 0.45 & 8.3 & 0.45 & 8.2 \\
\addlinespace[3pt]
  \texttt{gemini-3.1-pro} & \textsc{zero-shot} & 0.55 & 5.7 & 0

In [5]:
# Also write the LaTeX to a file for easy inclusion in the paper
import os
os.makedirs('figures', exist_ok=True)
with open('figures/appendix_table.tex', 'w') as f:
    f.write(latex)
print('Saved to figures/appendix_table.tex')

Saved to figures/appendix_table.tex
